<h5> Imports de librerías de pyspark </h5>

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import *
from typing import Dict, List

In [0]:
input_catalog = "silver"
input_table_name = "total_sales"
output_catalog = "gold"
schema = "sales"
table_name = f"{output_catalog}.{schema}.total_sales_fact"
force_history = False

<h5> Configuración de la aplicación de Spark </h5>

In [0]:
spark = (
    SparkSession.builder
    .appName('SalesPipelineGoldApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

In [0]:
sales_silver_df = spark.table(f"{input_catalog}.{schema}.{input_table_name}")

In [0]:
from pyspark.sql.functions import (
    col, max as spark_max, min as spark_min, count, sum as spark_sum, 
    avg, datediff, current_date, when, lit, round as spark_round,
    ntile, concat_ws
)

from pyspark.sql.window import Window

# Calcular fecha de referencia (última fecha en los datos)
max_date_row = sales_silver_df.select(spark_max(col("FECHA")).alias("max_date")).collect()
reference_date = max_date_row[0]["max_date"] if max_date_row else current_date()


# Análisis RFM por cliente
rfm_analysis = (
    sales_silver_df
    .groupBy("IDCLIENTE", "NOMBRE_Y_APELLIDO", "EDAD", "PROVICIA_CLIENTE", "LOCALIDAD_CLIENTE")
    .agg(
        # Recency: días desde la última compra
        datediff(lit(reference_date), spark_max("FECHA")).alias("recency_days"),
        
        # Frequency: cantidad de compras
        count("IDVENTA").alias("frequency"),
        
        # Monetary: valor total gastado
        spark_sum(col("PRECIO") * col("CANTIDAD")).alias("monetary_value"),
        
        # Métricas adicionales
        avg(col("PRECIO") * col("CANTIDAD")).alias("avg_purchase_value"),
        spark_min("FECHA").alias("first_purchase_date"),
        spark_max("FECHA").alias("last_purchase_date"),
        count("IDPRODUCTO").alias("total_products_bought")
    )
)

# Calcular percentiles para segmentación RFM
window_spec = Window.orderBy(col("recency_days"))
rfm_with_scores = (
    rfm_analysis
    # Recency Score (invertido: menor días = mejor score)
    .withColumn("r_score", 6 - ntile(5).over(Window.orderBy(col("recency_days"))))
    # Frequency Score
    .withColumn("f_score", ntile(5).over(Window.orderBy(col("frequency"))))
    # Monetary Score
    .withColumn("m_score", ntile(5).over(Window.orderBy(col("monetary_value"))))
)

# Crear segmentos RFM
rfm_segmented = (
    rfm_with_scores
    .withColumn("rfm_score", concat_ws("-", col("r_score"), col("f_score"), col("m_score")))
    .withColumn(
        "customer_segment",
        when((col("r_score") >= 4) & (col("f_score") >= 4) & (col("m_score") >= 4), "Champions")
        .when((col("r_score") >= 3) & (col("f_score") >= 3) & (col("m_score") >= 3), "Loyal Customers")
        .when((col("r_score") >= 4) & (col("f_score") <= 2), "New Customers")
        .when((col("r_score") >= 3) & (col("f_score") <= 2) & (col("m_score") <= 2), "Promising")
        .when((col("r_score") <= 2) & (col("f_score") >= 3) & (col("m_score") >= 3), "At Risk")
        .when((col("r_score") <= 2) & (col("f_score") <= 2), "Lost Customers")
        .when((col("m_score") >= 4), "Big Spenders")
        .otherwise("Regular")
    )
)

# Segmentación por edad
rfm_with_age_segment = (
    rfm_segmented
    .withColumn(
        "age_segment",
        when(col("EDAD") < 25, "18-24")
        .when((col("EDAD") >= 25) & (col("EDAD") < 35), "25-34")
        .when((col("EDAD") >= 35) & (col("EDAD") < 45), "35-44")
        .when((col("EDAD") >= 45) & (col("EDAD") < 55), "45-54")
        .when((col("EDAD") >= 55) & (col("EDAD") < 65), "55-64")
        .when(col("EDAD") >= 65, "65+")
        .otherwise("Unknown")
    )
)

# Calcular días como cliente y tasa de retención
customer_segmentation_final = (
    rfm_with_age_segment
    .withColumn(
        "customer_lifetime_days",
        datediff(col("last_purchase_date"), col("first_purchase_date"))
    )
    .withColumn(
        "purchase_frequency_rate",
        when(col("customer_lifetime_days") > 0,
             spark_round(col("frequency") / (col("customer_lifetime_days") / 30), 2))
        .otherwise(col("frequency"))
    )
    .withColumn(
        "is_active",
        when(col("recency_days") <= 90, lit(True)).otherwise(lit(False))
    )
    .withColumn("CREATED_AT", current_timestamp())
    .withColumn("UPDATED_AT", current_timestamp())
)

# Mostrar resumen
display(customer_segmentation_final.orderBy(col("monetary_value").desc()))

<h4> Ordenar columnas </h4>

##### Preparamos dataframe para generar tabla en silver
<p>Convertimos todas las columnas a mayúsculas</p> 
<p>Creamos las fechas de sistema</p>

<h4> Carga de datos </h4>

In [0]:
try: 
    processed_df.write.format("delta").saveAsTable(f"{table_name}")
except AnalysisException as err:
    if "TABLE_OR_VIEW_ALREADY_EXISTS" in err.getErrorClass():
        exclude_update = ['CREATED_AT', 'UPDATED_AT','R_HASH']
        target_delta_table = DeltaTable.forName(spark, f"{table_name}")
        target_df = target_delta_table.toDF()
        source_df = processed_df
        delta_merge_builder = (
                target_delta_table.alias('target')
                .merge(
                    source_df.alias('source'), 
                    "target.PK_HASH = source.PK_HASH"
                )
                .whenMatchedUpdate(
                    set = {f"target.{col}" : f"source.{col}" for col in source_df.columns if col != exclude_update}
                )
                .whenNotMatchedInsertAll()
            )
        delta_merge_builder.execute()
    else:
        raise err